# Fetch TCEQ Central Records by Proj Number

Source: https://records.tceq.texas.gov/

TCEQ Records Online runs on an Oracle Content Server. The home page redirects to
the search service (`/cs/idcplg?IdcService=TCEQ_SEARCH`) and the search form is
just a `GET` to `/cs/idcplg`. To pull the files for a project we search on its
`Secondary ID`, then download the documents we care about.

In [ ]:
import re
import time
from io import StringIO
from pathlib import Path
from urllib.parse import unquote, urljoin

import pandas as pd
import requests
from lxml import html

BASE_URL = 'https://records.tceq.texas.gov/'
ENDPOINT = BASE_URL + 'cs/idcplg'

# Create project root
def find_project_root(marker: str = '.git') -> Path:
    '''Climbs up folders starting from CWD until it finds the root.'''
    current = Path.cwd().resolve()

    # Loop upward until hitting the project root (where current == its own parent)
    while current != current.parent:
        if (current / marker).exists():
            return current
        current = current.parent

PROJECT_ROOT = find_project_root(marker='.git')
OUT_DIR = PROJECT_ROOT / 'data' / 'raw' / 'records'

In [ ]:
DROP_COLS = ['Spacer Column', 'Select', 'Actions']


def _download_urls(page_html: str) -> list:
    '''
    Per-row link to the actual file.

    The dID buried in the link isn't the Content ID shown in the table, so we
    grab it straight from the <a href> instead of rebuilding it ourselves.
    '''
    tbl = html.fromstring(page_html).get_element_by_id('table_0')
    rows = tbl.xpath('.//tr')
    header = [' '.join(c.xpath('.//text()')).strip() for c in rows[0]]
    cid = header.index('Content ID')

    urls = []
    for row in rows[1:]:
        hrefs = row[cid].xpath('.//a/@href')
        urls.append(urljoin(BASE_URL, hrefs[0]) if hrefs else None)
    return urls


def fetch(secondary_id: int, session: requests.Session) -> pd.DataFrame:
    # The service fails unless the whole form goes with it, blank fields and all.
    # select0/input0 is the one criterion we actually set.
    params = {
        'IdcService': 'TCEQ_PERFORM_SEARCH',
        'xIdcProfile': 'Record',
        'IsExternalSearch': '1',
        'sortSearch': 'false',
        'newSearch': 'true',
        'xRecordSeries': '0',
        'xInsightDocumentType': '0',
        'xMedia': '0',
        'operator': 'AND',
        'select0': 'xSecondaryID', 'input0': str(secondary_id),
        'select1': '', 'input1': '',
        'select2': '', 'input2': '',
        'select3': '', 'input3': '',
        'ftx': '',
    }
    resp = session.get(ENDPOINT, params=params, timeout=120)
    resp.raise_for_status()

    # No matches -> the table isn't rendered and read_html raises
    try:
        raw = pd.read_html(StringIO(resp.text), attrs={'id': 'table_0'})[0]
    except ValueError:
        return pd.DataFrame()

    raw.columns = raw.iloc[0]                  # First row is the column names
    df = raw.iloc[1:].drop(columns=DROP_COLS)
    df['Download URL'] = _download_urls(resp.text)
    return df

## Download a record's file

Each row links to its file through `TCEQ_EXTERNAL_SEARCH_GET_FILE`. `download_file`
follows the link and saves it under whatever name the server hands back.

In [ ]:
def _filename_from_response(resp: requests.Response) -> str:
    '''
    Filename the server hands back in Content-Disposition (RFC 5987 encoded),
    falling back to the dID when it's missing.
    '''
    cd = resp.headers.get('Content-Disposition', '')
    match = re.search(r"filename\*=UTF-8''([^;]+)", cd) or re.search(r'filename="?([^";]+)', cd)
    if match:
        return unquote(match.group(1)).strip()
    return f"record-{resp.url.split('dID=')[-1].split('&')[0]}.pdf"


def download_file(url: str, session: requests.Session, out_dir: Path = OUT_DIR) -> Path:
    resp = session.get(url, timeout=180)
    resp.raise_for_status()

    out_dir.mkdir(parents=True, exist_ok=True)
    dest = out_dir / _filename_from_response(resp)
    dest.write_bytes(resp.content)
    return dest

## Extract Relevant Proj Num

In [ ]:
CANDIDATES_CSV = PROJECT_ROOT / 'data' / 'processed' / 'candidates-list.csv'

# Project Number is the same value the records search calls the Secondary ID
def load_project_numbers(csv_path: Path = CANDIDATES_CSV) -> list:
    projects = pd.read_csv(csv_path, usecols=['Project Number'])['Project Number']
    return projects.dropna().astype(int).unique().tolist()

CANDIDATE_IDS = load_project_numbers()
print(f'{len(CANDIDATE_IDS)} project numbers')

## Configure & run

Loop the project numbers and, for each, grab the record whose Title is
`DOWNLOAD_TITLE` (e.g. Technical Review) into `data/raw/records/`. The results
table is only there to find the download link, we don't keep it.

In [ ]:
# Change values here for desired scope
SECONDARY_IDS = CANDIDATE_IDS
DOWNLOAD_TITLE = 'Technical Review'   # set to None to skip the downloads
SLEEP_SECONDS = 1.0

OUT_DIR.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers['User-Agent'] = 'Mozilla/5.0'

for secondary_id in SECONDARY_IDS:
    df = fetch(secondary_id, session)
    print(f'Secondary ID {secondary_id} -> {len(df):>4} rows')

    if DOWNLOAD_TITLE and not df.empty:
        for url in df.loc[df['Title'] == DOWNLOAD_TITLE, 'Download URL']:
            saved = download_file(url, session)
            print(f'    downloaded  {saved.relative_to(PROJECT_ROOT)}')
    time.sleep(SLEEP_SECONDS)